# Session 1: can a computer guess the formula?
**A. Real-world problem.** Drop a ball and note how far it has fallen after each second.
Galileo did this with rolling balls and found a law. Today a computer can search for
the law by trying many formulas and keeping the one that matches the data best.

**Plan.** (1) make fall data that follow a law with a little measurement noise,
(2) write a few candidate formulas with one unknown number c each, (3) fit c for each,
(4) score each by its error, (5) read the winner and what it tells us about gravity g.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the data
A ball dropped from rest falls a distance d = (1/2) g t^2 metres after t seconds,
where g is the acceleration due to gravity (9.81 m/s^2 on Earth). Real measurements are
never exact, so we multiply each distance by (1 + a small error). The six errors below
are fixed numbers (2%, -3%, 1%, -2%, 3%, -1%) so that everyone gets the same data.

In [ ]:
T_OBS = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])             # seconds
REL_ERR = np.array([0.02, -0.03, 0.01, -0.02, 0.03, -0.01])  # fractional measurement errors


def fall_distance(t, g=9.81):
    """True distance fallen (metres) after t seconds, starting from rest."""
    return 0.5 * g * np.asarray(t, float) ** 2


def fall_data(g=9.81):
    """Measured distances (metres) at T_OBS, with the fixed measurement errors."""
    return fall_distance(T_OBS, g) * (1.0 + REL_ERR)

## Step 2: a library of candidate formulas
Each formula has the form d = c * f(t), where f is a shape (t, t squared, ...) and c is
one unknown number. We can find the best c with a formula. Minimising the sum of
squared differences gives
    c = sum(f * d) / sum(f * f).
The score is the mean squared error (MSE): the average of (data - formula)^2, in metres squared.

In [ ]:
LIBRARY = {
    "c * sqrt(t)": np.sqrt,
    "c * t": lambda t: t,
    "c * t^2": lambda t: t ** 2,
    "c * t^3": lambda t: t ** 3,
    "c * exp(t)": np.exp,
}


def fit_c(shape, d):
    """Best c and mean squared error for d = c * shape."""
    shape = np.asarray(shape, float)
    c = float(np.sum(shape * d) / np.sum(shape * shape))
    mse = float(np.mean((d - c * shape) ** 2))
    return c, mse


def search_library(t, d):
    """Try every formula in LIBRARY; return a list of (name, c, mse) sorted by mse."""
    rows = [(name, *fit_c(f(t), d)) for name, f in LIBRARY.items()]
    return sorted(rows, key=lambda r: r[2])

## Step 3: a wider search over powers
Instead of a short list, try d = c * t^n for many exponents n from 0.5 to 3.5. The
exponent with the lowest error is the computer's guess for the shape of the law.

In [ ]:
def best_exponent(t, d, exponents=None):
    """Return (n, c, mse) for the exponent n with the lowest mse, and the whole curve of mse."""
    if exponents is None:
        exponents = np.arange(0.5, 3.51, 0.1)
    curve = [fit_c(np.asarray(t, float) ** n, d)[1] for n in exponents]
    i = int(np.argmin(curve))
    c, mse = fit_c(np.asarray(t, float) ** exponents[i], d)
    return float(exponents[i]), c, mse, np.array(curve), exponents


def gravity_from_c(c):
    """If d = c t^2 then c = g/2, so g = 2 c (m/s^2)."""
    return 2.0 * c

## Step 4: see it
Left: the data and three candidate curves. Right: error against the exponent n; the
deep valley sits at n = 2.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    d = fall_data()
    for name, c, mse in search_library(T_OBS, d):
        print(f"{name:12s} c = {c:8.3f}   MSE = {mse:10.3f} m^2")
    n, c, mse, curve, exps = best_exponent(T_OBS, d)
    print(f"best exponent n = {n:.1f}, c = {c:.3f}, so g = {gravity_from_c(c):.2f} m/s^2")

    tt = np.linspace(0, 3.2, 200)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(T_OBS, d, "ko", label="measured data")
    for name, f in (("c * t", LIBRARY["c * t"]), ("c * t^2", LIBRARY["c * t^2"]), ("c * t^3", LIBRARY["c * t^3"])):
        cc, _ = fit_c(f(T_OBS), d)
        ax.plot(tt, cc * f(tt), label=name)
    ax.set_xlabel("time t (s)")
    ax.set_ylabel("distance fallen d (m)")
    ax.set_title("Which formula follows the falling ball?")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "fall_candidates.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.semilogy(exps, curve, "o-")
    ax2.set_xlabel("exponent n in d = c t^n")
    ax2.set_ylabel("mean squared error (m^2)")
    ax2.set_title("Error valley: the best exponent is 2")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "error_vs_exponent.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change the gravity in `fall_data` to 1.62 (the Moon) and run the library search. Predict the winner first and the value of g you get.
2. **Stretch.** Add the candidate `c * t^2.5` and `c * log(1 + t)` to LIBRARY and see where they rank.
3. **Explain.** Explain in two sentences why the exponent search is more informative than the short library.

## Assessment
- **A. Conceptual.** Two exponents, n = 1.9 and n = 2.1, give errors of 0.37 and 0.48 m^2, close to the 0.20 m^2 of n = 2. Can the data prove that n is exactly 2?
- **B. Numerical.** A shape f = (1, 2) is fitted to the data d = (3, 5) with d = c f. Find c and the MSE by hand.
- **C. Graph and simulation interpretation.** On the error-versus-exponent curve, the error at n = 1 is 34.39 m^2 and at n = 3 it is 15.73 m^2. Which wrong guess is worse, and where is the minimum?
- **D. Design and optimisation challenge.** Design: you can time only four drops with a stopwatch accurate to about 0.1 s. Choose the drop times (or heights) and say how you would decide between n = 1.5, 2 and 3, and what would make your decision more reliable.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("a ball dropped for 2 s on Earth falls 19.62 m (worked example, noise-free)",
                   abs(float(L.fall_distance(2.0)) - 19.62) < 1e-9))
    d = L.fall_data()
    c, mse = L.fit_c(L.T_OBS ** 2, d)
    checks.append(("worked example: fitted c = 4.907 and mean squared error 0.205 m^2 for d = c t^2",
                   round(c, 3) == 4.907 and round(mse, 3) == 0.205))
    checks.append(("worked example: g = 2c is 9.81 m/s^2 to 3 significant figures",
                   round(L.gravity_from_c(c), 2) == 9.81))
    checks.append(("worked example: the six measured distances are 1.251, 4.758, 11.147, 19.228, 31.576, 43.704 m",
                   np.allclose(d, [1.251, 4.758, 11.147, 19.228, 31.576, 43.704], atol=5e-4)))
    checks.append(("worked example: sum(f d) = 697.742 and sum(f f) = 142.1875, so c = 4.907",
                   abs(float(np.sum(L.T_OBS ** 2 * d)) - 697.742) < 5e-4 and abs(float(np.sum(L.T_OBS ** 4)) - 142.1875) < 1e-9))
    cm, _ = L.fit_c(L.T_OBS ** 2, L.fall_data(1.62))
    checks.append(("on the Moon (g = 1.62) the same measurement errors give an estimated g of 1.62",
                   round(L.gravity_from_c(cm), 2) == 1.62))
    checks.append(("errors at n = 1.9 and 2.1 are 0.37 and 0.48 m^2",
                   round(L.fit_c(L.T_OBS ** 1.9, d)[1], 2) == 0.37 and round(L.fit_c(L.T_OBS ** 2.1, d)[1], 2) == 0.48))
    c_hand, mse_hand = L.fit_c(np.array([1.0, 2.0]), np.array([3.0, 5.0]))
    checks.append(("hand calculation: shape (1, 2), data (3, 5) gives c = 13/5 = 2.6 and MSE 0.1",
                   abs(c_hand - 2.6) < 1e-12 and abs(mse_hand - 0.1) < 1e-12))
    rows = L.search_library(L.T_OBS, d)
    checks.append(("the library search ranks c * t^2 first, and the runner-up has 20 times more error",
                   rows[0][0] == "c * t^2" and rows[1][2] > 20 * rows[0][2]))
    n, c2, m2, curve, exps = L.best_exponent(L.T_OBS, d)
    checks.append(("the exponent search finds n = 2.0", abs(n - 2.0) < 1e-9))
    checks.append(("errors at n = 1, 1.5, 2, 2.5, 3 are 34.39, 6.71, 0.20, 5.00, 15.73 m^2",
                   [round(L.fit_c(L.T_OBS ** k, d)[1], 2) for k in (1.0, 1.5, 2.0, 2.5, 3.0)] == [34.39, 6.71, 0.20, 5.00, 15.73]))
    checks.append(("with no noise the t^2 formula fits exactly (MSE below 1e-20)",
                   L.fit_c(L.T_OBS ** 2, L.fall_distance(L.T_OBS))[1] < 1e-20))
    checks.append(("edge: at t = 0 the ball has not fallen, and on the Moon (g = 1.62) 2 s gives 3.24 m",
                   float(L.fall_distance(0.0)) == 0.0 and abs(float(L.fall_distance(2.0, 1.62)) - 3.24) < 1e-9))
    checks.append(("the measured data differ from the law by at most 3 percent",
                   float(np.max(np.abs(d / L.fall_distance(L.T_OBS) - 1))) <= 0.03 + 1e-12))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
